# Murano retirement mailout

Finds every Murano environment with a live instance, groups them by project, and emails each project's Tenant Managers and Members with their own deployments listed.

Tracked in NECTAR-16917 (contacts) and NECTAR-16921 (comms).

In [ ]:
from collections import defaultdict

from openstack import exceptions as os_exc

from mailout_helper import MailoutHelper

## Dates and links

Option A timeline. Confirm before sending.

In [ ]:
FREEZE_DATE = 'Tuesday 1 December 2026'
SHUTDOWN_DATE = 'Wednesday 30 June 2027'
DOCS_URL = 'https://support.ehelp.edu.au/'  # TODO: Murano migration article (NECTAR-16920)

# Server states that count as a live instance worth telling the owner about.
LIVE_STATUSES = {'ACTIVE', 'SHUTOFF', 'SHELVED', 'SHELVED_OFFLOADED',
                 'PAUSED', 'SUSPENDED', 'ERROR'}

## Set up OpenStack

Needs an admin token: it lists environments across all projects and authenticates to Taynac.

In [ ]:
helper = MailoutHelper()
helper.setup_openstack('https://identity.rc.nectar.org.au/')

# Murano's all_tenants listing needs the 'admin' role on the token's project
# (prod policy admin_api: 'role:admin or is_admin:True'), else HTTP 403.
auth_ref = helper.conn.session.auth.get_auth_ref(helper.conn.session)
print(f'Scoped to project: {auth_ref.project_name} ({auth_ref.project_id})')
print(f'Roles: {sorted(auth_ref.role_names)}')
if 'admin' not in auth_ref.role_names:
    print("WARNING: no 'admin' role on this token; the environment listing will be refused")

## Murano API access

openstacksdk has no Murano proxy, so call the REST API through the authenticated session.

In [ ]:
MURANO = {'service_type': 'application-catalog', 'interface': 'public'}


def murano_get(path, **params):
    r = helper.conn.session.get(path, endpoint_filter=MURANO, params=params)
    r.raise_for_status()
    return r.json()


def walk_instances(services):
    """(app, server id) for every io.murano.resources.*Instance in an env."""
    found = []

    def walk(obj, owner):
        if isinstance(obj, dict):
            hdr = obj.get('?')
            t = hdr.get('type', '').split('/')[0] if isinstance(hdr, dict) else ''
            if t and not t.startswith('io.murano'):
                owner = t.split('.')[-1]
            if t.startswith('io.murano.resources') and 'Instance' in t and obj.get('openstackId'):
                found.append((owner, obj['openstackId']))
            for v in obj.values():
                walk(v, owner)
        elif isinstance(obj, list):
            for v in obj:
                walk(v, owner)

    walk(services, None)
    return found

## Find environments with live instances

About 900 environments, so this takes a while.

In [ ]:
envs = murano_get('/v1/environments', all_tenants='true')['environments']
print(f'{len(envs)} environments')

by_project = defaultdict(list)
for e in envs:
    if e['name'].startswith('Test_Murano_'):
        continue  # tempest
    detail = murano_get(f"/v1/environments/{e['id']}")
    live = []
    for app, sid in walk_instances(detail.get('services', [])):
        try:
            s = helper.conn.compute.get_server(sid)
        except os_exc.NotFoundException:
            continue
        if s.status in LIVE_STATUSES:
            live.append({'app': app, 'id': s.id, 'name': s.name, 'status': s.status})
    if live:
        by_project[e['tenant_id']].append({'name': e['name'], 'instances': live})

print(f'{len(by_project)} projects with live instances')

## Project and recipient data

In [ ]:
data = {}
for project_id, environments in by_project.items():
    project = helper.get_project(project_id)
    if not project.is_enabled:
        print(f'Skipping disabled project {project.name}')
        continue
    data[project_id] = {
        'project': project,
        'managers': helper.get_tenant_managers(project_id),
        'members': helper.get_project_members(project_id, exclude_disabled=True),
        'environments': environments,
        'freeze_date': FREEZE_DATE,
        'shutdown_date': SHUTDOWN_DATE,
        'docs_url': DOCS_URL,
    }
print(f'{len(data)} projects to notify')

## Generate notifications

In [ ]:
subject = 'Nectar Application Catalog service retirement: action needed for project {{ project.name }}'
body = 'murano-retirement-notification.j2'

notifications = helper.generate_notifications_from_instances(data, subject, body)
no_recipient = [n for n in notifications if not n['to']]
print(f'Generated {len(notifications)} notifications, {len(no_recipient)} without a recipient')

## Preview

In [ ]:
helper.preview_notification(notifications[0])

## Send

Send line is commented out. Test on one notification first.

In [ ]:
for notification in notifications[:1]:
    if not notification['to']:
        continue
    print(f"Sending to {notification['to']} and {len(notification.get('cc', []))} others...")
    #helper.send_notification(notification)

print('Finished')